# Towards Generative Modeling of CO₂ Geological Storage with Latent Conditional Diffusion Models

|| **Author: Vittoria De Pellegrini** || **Year: 2025** ||
**Institution: King Abdullah University of Science and Technology (KAUST)** ||

This notebook provides a comprehensive guide for testing a **VQ-VAE (Vector Quantized Variational Autoencoder) + GAN** Model for CO₂ Gas Saturation Dataset.

### ⚠️ Define Normalization Strategy

To normalize our dataset to the range $[-1, 1]$, we utilize **Local Normalization**. This approach involves normalizing each sample independently based on its own minimum and maximum values. We adopt local normalization because the range of values differs between samples. The formula for local normalization is calculated as follows: 

$$x_{\text{norm}} = 2 \times \frac{x - \min(x)}{\max(x) - \min(x)} - 1\tag{1}$$

Where:
- $\min(x)$ and $\max(x)$ are the minimum and maximum values of the each sample $x$ in the dataset $X = \{x_1, x_2, \dots, x_N\}$, where $N$ is the number of samples.
- $x_{\text{norm}}$ denotes the normalized value, which will falls in the range $[-1, 1]$.

Note: 
- For the Condition (porosity) we still apply **Global Normalization**, and we use it for plotting purposes.

### 📚⬇ Import Libraries and Functions 

In [1]:
# Import libraries 
import warnings
warnings.filterwarnings('ignore')

import os 
import torch
from torch.utils.data import TensorDataset

import sys
sys.path.append('/.../')

# Import functions 
from dmfco2 import(
    set_seed,  
    VQVAE,   
    visualize_vqvae_reconstructed_co2_samples_test
)

### ❗️ Identify Number of Devices (Multi-GPU Training with Accelerate)

In [2]:
# Print the number of GPUs available
num_devices = torch.cuda.device_count()
print("Number of devices: ", num_devices)

# Loop through all available devices and print their names
for i in range(num_devices):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
    
# Print current device
print("Current device: ", torch.cuda.current_device())

### ❗️Set Seed for Reproducible Results

In [3]:
# Set seed
set_seed(seed=42)

### 📙 Define Inputs (Conditions) and Outputs (Targets) Variables for CO₂ Gas Saturation Test Dataset

In [4]:
# Define validation dataset directory path
data_dir_sg_test = '/.../co2_dataset/test_dataset_gas_saturation'

In [5]:
# Define inputs (conditions) and outputs (targets) variables
test_sg_inputs = torch.load(f'{data_dir_sg_test}/sg_test_inputs.pt')
test_sg_outputs = torch.load(f'{data_dir_sg_test}/sg_test_outputs.pt')

print(test_sg_inputs.shape) # Shape: [num_samples, height, width, time_steps, num_features]
print(test_sg_outputs.shape)

In [6]:
# Inputs -> Extract porosity (index 2) and last time frame 30.0 y (index -1)
test_sg_inputs  = test_sg_inputs[:, :, :, -1, 2]  

# Outputs -> Extract last time frame 30.0 y
test_sg_outputs = test_sg_outputs[:, :, :, -1] 

print("New shape:", test_sg_inputs.shape)
print("New shape:", test_sg_outputs.shape)

In [7]:
# Add number of features (1) to inputs (conditions) and outputs (targets)
test_sg_inputs = test_sg_inputs[:, :, :].unsqueeze(1) # Shape will be [500, 1, 96, 200]
test_sg_outputs = test_sg_outputs[:, :, :].unsqueeze(1) # Shape will be [500, 1, 96, 200]

# Print new shape
print("New shape:", test_sg_inputs.shape)   # Expected: [num_samples, num_features, height, width] -> Only Porosity as Condition 
print("New shape:", test_sg_outputs.shape)  # Expected: [num_samples, num_features, height, width] -> Only CO₂ Gas Saturation @ year 30.0

In [8]:
# Check ranges for each sample individually (CONDITION)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_test_por_individual = test_sg_inputs.amin(dim=(1, 2, 3))  # Min per sample
max_test_por_individual = test_sg_inputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_test_por_individual: {min_test_por_individual.shape}") # [500]
print(f"Shape of max_test_por_individual: {max_test_por_individual.shape}") # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_test_por_individual, max_test_por_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")

In [9]:
# Normalize inputs between -1 and 1 
min_test_por = test_sg_inputs.min()
max_test_por = test_sg_inputs.max()

# Check min and max values in test_sg_inputs before global normalization
print("Global Min for Porosity before normalization:", test_sg_inputs.min().item(), "Global Max for Porosity before normalization:", test_sg_inputs.max().item())

# Global Normalization formula
test_sg_inputs = 2 * (test_sg_inputs - min_test_por) / (max_test_por - min_test_por) - 1

# Check min and max values in test_sg_inputs after global normalization
print("Global Min for Porosity after normalization:", test_sg_inputs.min().item(), "Global Min for Porosity after normalization:", test_sg_inputs.max().item())

In [10]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_test_sg_individual = test_sg_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_test_sg_individual = test_sg_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_test_sg_individual: {min_test_sg_individual.shape}")  # [500]
print(f"Shape of max_test_sg_individual: {max_test_sg_individual.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_test_sg_individual, max_test_sg_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in test_sg_outputs before local normalization
print("Global Min for CO₂ Gas Saturation before normalization:", test_sg_outputs.min().item(), "Global Max for CO₂ Gas Saturation before normalization:", test_sg_outputs.max().item())

In [11]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
co2_log_file = os.path.join(log_dir, "test_co2_min_max_before_normalization.txt")

# Save min and max values before normalization into a text file
with open(co2_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values Before Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_test_sg_individual)):
        log_file.write(f"Sample {i+1}: Min Before={min_test_sg_individual[i].item():.4f}, Max Before={max_test_sg_individual[i].item():.4f}\n")

In [12]:
# Normalize outputs between -1 and 1 

# Initialize arrays to store min and max values for each sample
min_test_sg = []
max_test_sg = []

# Normalize outputs and store min/max values
for i in range(len(test_sg_outputs)):
    min_val = test_sg_outputs[i].min()
    max_val = test_sg_outputs[i].max()
    print(f"Sample {i + 1}:")
    print(f"Original Min: {min_val}, Original Max: {max_val}")
    min_test_sg.append(min_val)
    max_test_sg.append(max_val)

    # Local Normalization formula
    test_sg_outputs[i] = 2 * (test_sg_outputs[i] - min_val) / (max_val - min_val) - 1
    print(f"Normalized Output: {test_sg_outputs[i]}")

In [13]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_test_sg_individual_after_norm = test_sg_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_test_sg_individual_after_norm = test_sg_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_test_sg_individual_after_norm: {min_test_sg_individual_after_norm.shape}")  # [500]
print(f"Shape of max_test_sg_individual_after_norm: {max_test_sg_individual_after_norm.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_test_sg_individual_after_norm, max_test_sg_individual_after_norm)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in test_sg_outputs after local normalization
print("Global Min for CO₂ Gas Saturation after normalization:", test_sg_outputs.min().item(), "Global Min for CO₂ Gas Saturation after normalization:", test_sg_outputs.max().item())

In [14]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
co2_log_file = os.path.join(log_dir, "test_co2_min_max_after_normalization.txt")

# Save min and max values before normalization into a text file
with open(co2_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values After Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_test_sg_individual_after_norm)):
        log_file.write(f"Sample {i+1}: Min After={min_test_sg_individual_after_norm[i].item():.4f}, Max After={max_test_sg_individual_after_norm[i].item():.4f}\n")

In [15]:
# Define variables for conditional diffusion model
test_sg_label = test_sg_inputs
test_sg_target = test_sg_outputs

### 🗂️ Define Test Dataset

In [16]:
# Create dataloaders 
testing_dataset = TensorDataset(test_sg_target, test_sg_label)

### 🚀 Load Pre-Trained VQ-VAE + GAN Model

In [17]:
# Load saved model
vqvae_checkpoint = '/.../trained_models/gas_saturation_models/vqvae/best_model/vqvae_model.pth' 
vqvae_checkpoint = torch.load(vqvae_checkpoint)

# Extract parameters and weights
z_channels = vqvae_checkpoint["z_channels"]
codebook_size = vqvae_checkpoint["codebook_size"]
state_dict = vqvae_checkpoint["state_dict"]

# Initialize the model with the saved parameters
vqvae_model = VQVAE(z_channels=z_channels, codebook_size=codebook_size)

# Load the model weights
vqvae_model.load_state_dict(state_dict)

print("Model loaded successfully with z_channels={} and codebook_size={}".format(z_channels, codebook_size))

### 🔗 Save Reconstructed Samples

In [ ]:
# Define save directory for loss curves
save_dir = "/.../results/vqvae_gas_saturation_results/reconstructed_samples_test_dataset"

# Specify the first 500 sample indices
sample_indices = list(range(500))

# Call the function for CO₂ gas saturation
visualize_vqvae_reconstructed_co2_samples_test(
    model=vqvae_model,
    dataset=testing_dataset,
    sample_indices=sample_indices,
    min_test_sg=min_test_sg,
    max_test_sg=max_test_sg,
    min_test_por=min_test_por,
    max_test_por=max_test_por,
    save_dir=save_dir
)

## References 

**[1] Wen et al., 2002** ([Paper](https://www.sciencedirect.com/science/article/pii/S0309170822000562)): CO₂ Open-Source Dataset ([Google Drive Link](https://drive.google.com/drive/folders/1fZQfMn_vsjKUXAfRV0q_gswtl8JEkVGo)).